In [5]:
import asyncio
import json
import uuid
import time
from datetime import datetime
from dataclasses import dataclass, field, asdict
from enum import Enum, auto
from typing import Dict, List, Optional, Callable
import random

In [ ]:
import asyncio
import json
import uuid
import time
from datetime import datetime
from dataclasses import dataclass, field, asdict
from enum import Enum, auto
from typing import Dict, List, Optional, Callable
import random

# --- 1. INFRAESTRUTURA SIMULADA (Broker MQTT & State Store) ---
# Em produção: Eclipse Mosquitto (Broker) + Redis (State) + PostgreSQL (Audit)

class MessageType(Enum):
    TELEMETRY = "telemetry/data"
    COMMAND = "fleet/command"
    ALERT = "fleet/alert"
    METRIC = "system/metric"

class VehicleState(Enum):
    OPERATIONAL = "OPERATIONAL"
    WARNING = "WARNING"
    CRITICAL_STOP = "CRITICAL_STOP"
    MAINTENANCE = "MAINTENANCE"
    OFFLINE = "OFFLINE"

@dataclass
class Message:
    topic: str
    payload: dict
    timestamp: float = field(default_factory=time.time)
    message_id: str = field(default_factory=lambda: str(uuid.uuid4()))

class MockMQTTBroker:
    """Simula um broker MQTT distribuído com pub/sub."""
    def __init__(self):
        self.subscribers: Dict[str, List] = {}
        self.message_queue = asyncio.Queue()

    def subscribe(self, topic: str, callback: Callable):
        if topic not in self.subscribers:
            self.subscribers[topic] = []
        self.subscribers[topic].append(callback)
        print(f"📡 Subscriber registrado no tópico: {topic}")

    async def publish(self, topic: str, payload: dict):
        msg = Message(topic=topic, payload=payload)
        # Simula latência de rede do broker
        await asyncio.sleep(random.uniform(0.005, 0.02))
        if topic in self.subscribers:
            tasks = [cb(msg) for cb in self.subscribers[topic]]
            await asyncio.gather(*tasks, return_exceptions=True)

class StateStore:
    """Simula um Redis para estado volátil de alta velocidade."""
    def __init__(self):
        self.store: Dict[str, dict] = {}

    async def get(self, key: str) -> Optional[dict]:
        return self.store.get(key)

    async def set(self, key: str, value: dict):
        self.store[key] = value

    async def update_state(self, vehicle_id: str, new_state: VehicleState):
        if vehicle_id not in self.store:
            self.store[vehicle_id] = {"state": VehicleState.OPERATIONAL.value, "history": []}

        old_state = self.store[vehicle_id]["state"]
        if old_state != new_state.value:
            self.store[vehicle_id]["state"] = new_state.value
            self.store[vehicle_id]["history"].append({
                "timestamp": datetime.now().isoformat(),
                "from": old_state,
                "to": new_state.value
            })
            return True # Estado mudou
        return False

# --- 2. DOMÍNIO DE NEGÓCIO (Regras Complexas) ---

class RiskEngine:
    """Motor de inferência estatística (substituto do LLM para baixa latência)."""

    @staticmethod
    def calculate_score(telemetry: dict) -> float:
        # Algoritmo ponderado multivariado
        temp_factor = max(0, (telemetry['temp'] - 90) / 30) # Normalizado 0-1+
        rpm_factor = max(0, (telemetry['rpm'] - 4000) / 1000)
        load_factor = telemetry['speed'] / 120.0

        # Peso crítico para temperatura
        score = -(temp_factor * 0.7 + rpm_factor * 0.2 + load_factor * 0.1)

        # Penalidade por inconsistência física (Temp alta + RPM baixo = Falha de refrigeração)
        if telemetry['temp'] > 105 and telemetry['rpm'] < 1500:
            score -= 0.5

        return score

class DecisionPolicy:
    """Políticas determinísticas baseadas em estado e risco."""

    @staticmethod
    async def decide(telemetry: dict, current_state: VehicleState, risk_score: float) -> dict:
        next_state = current_state
        commands = []
        reason = ""

        # Máquina de Estados Finita
        if risk_score < -0.4 or telemetry['temp'] > 115:
            if current_state != VehicleState.CRITICAL_STOP:
                next_state = VehicleState.CRITICAL_STOP
                reason = "Limiar crítico atingido. Risco de dano catastrófico."
                commands.append({
                    "type": "EMERGENCY_STOP",
                    "target": f"DRV-{telemetry['id']}",
                    "message": "🛑 PARADA DE EMERGÊNCIA AUTOMÁTICA. Desligue o motor.",
                    "priority": 1
                })
                commands.append({
                    "type": "DISPATCH_TOW_TRUCK",
                    "location": telemetry['location'],
                    "priority": 1
                })

        elif risk_score < -0.15 or telemetry['temp'] > 100:
            if current_state not in [VehicleState.CRITICAL_STOP, VehicleState.MAINTENANCE]:
                next_state = VehicleState.WARNING
                reason = "Anomalia detectada. Necessária intervenção preventiva."
                commands.append({
                    "type": "ROUTING_DIVERT",
                    "target": f"DRV-{telemetry['id']}",
                    "message": f"⚠️ Desvie para oficina mais próxima. Max 40km/h.",
                    "priority": 2
                })
                commands.append({
                    "type": "CREATE_MAINTENANCE_ORDER",
                    "severity": "HIGH",
                    "details": f"Temp: {telemetry['temp']}C, Risk: {risk_score}"
                })

        elif current_state == VehicleState.WARNING and risk_score > -0.05:
            next_state = VehicleState.OPERATIONAL
            reason = "Parâmetros normalizados. Retornando à operação padrão."
            commands.append({
                "type": "CLEAR_ALERT",
                "target": f"DRV-{telemetry['id']}",
                "message": "✅ Sistema normalizado. Pode retomar velocidade operativa.",
                "priority": 3
            })

        return {
            "next_state": next_state,
            "commands": commands,
            "reason": reason,
            "risk_score": risk_score
        }

# --- 3. AGENTE AUTÔNOMO (Consumer de Alto Throughput) ---

class AutonomousFleetAgent:
    def __init__(self, broker: MockMQTTBroker, store: StateStore):
        self.broker = broker
        self.store = store
        self.processed_count = 0
        self.error_count = 0
        self.start_time = time.time()

        # Registra-se como consumer
        self.broker.subscribe(MessageType.TELEMETRY.value, self.on_telemetry_received)
        print("🤖 Iniciado e ouvindo stream de telemetria...")

    async def on_telemetry_received(self, msg: Message):
        start_proc = time.time()
        try:
            data = msg.payload
            vid = data['id']

            # 1. Recuperar Estado Atual
            current_state_val = (await self.store.get(vid) or {}).get("state", VehicleState.OPERATIONAL.value)
            current_state = VehicleState[current_state_val]

            # 2. Calcular Risco (AI Inference)
            risk_score = RiskEngine.calculate_score(data)

            # 3. Tomar Decisão (Policy Engine)
            decision = await DecisionPolicy.decide(data, current_state, risk_score)

            # 4. Atualizar Estado (State Transition)
            state_changed = await self.store.update_state(vid, decision['next_state'])

            # 5. Executar Comandos (Se houver mudança ou ação crítica)
            if decision['commands']:
                for cmd in decision['commands']:
                    # Publica comando no tópico específico para atuadores/mobile
                    await self.broker.publish(MessageType.COMMAND.value, {
                        "vehicle_id": vid,
                        "command": cmd,
                        "triggered_by": msg.message_id,
                        "reason": decision['reason']
                    })

            # Log de Auditoria em Tempo Real
            if state_changed or decision['next_state'] == VehicleState.CRITICAL_STOP:
                audit_log = {
                    "event": "STATE_TRANSITION",
                    "vehicle_id": vid,
                    "old_state": current_state.value,
                    "new_state": decision['next_state'].value,
                    "risk_score": risk_score,
                    "commands_issued": len(decision['commands']),
                    "latency_ms": (time.time() - start_proc) * 1000
                }
                await self.broker.publish(MessageType.ALERT.value, audit_log)

            self.processed_count += 1

        except Exception as e:
            self.error_count += 1
            # Dead Letter Queue simulation
            await self.broker.publish("system/dlq", {"error": str(e), "original_msg": msg.payload})

    def get_metrics(self) -> dict:
        elapsed = time.time() - self.start_time
        return {
            "throughput_msgs_per_sec": self.processed_count / elapsed if elapsed > 0 else 0,
            "total_processed": self.processed_count,
            "errors": self.error_count,
            "uptime_sec": elapsed
        }

# --- 4. SIMULADOR DE FROTA (Producer) ---

class FleetSimulator:
    def __init__(self, broker: MockMQTTBroker):
        self.broker = broker
        self.vehicles = [
            {"id": f"V-{i:03d}", "base_temp": 85 + random.uniform(-2, 2), "base_rpm": 2000}
            for i in range(1, 11) # 10 veículos
        ]

    async def run(self, duration_sec=5):
        print(f"🚛 Iniciando frota com {len(self.vehicles)} veículos por {duration_sec}s...")
        end_time = time.time() + duration_sec

        while time.time() < end_time:
            for v in self.vehicles:
                # Gera variações realistas e injeta falhas aleatórias
                noise = random.uniform(-5, 5)
                fault_injection = 0
                if random.random() < 0.02: # 2% chance de falha súbita
                    fault_injection = random.uniform(20, 40)

                telemetry = {
                    "id": v['id'],
                    "timestamp": datetime.now().isoformat(),
                    "rpm": int(v['base_rpm'] + random.uniform(-200, 200)),
                    "speed": random.uniform(40, 90),
                    "temp": v['base_temp'] + noise + fault_injection,
                    "location": random.choice(["Rod. Anhanguera", "Centro SP", "Rod. Dutra", "Pátio"]),
                    "fuel_level": random.uniform(10, 100)
                }

                await self.broker.publish(MessageType.TELEMETRY.value, telemetry)

            # Controla a frequência de envio (ex: 10Hz por veículo)
            await asyncio.sleep(0.1)

# --- 5. ORQUESTRAÇÃO PRINCIPAL ---

async def main():
    print("🌐 INICIANDO SISTEMA DISTRIBUÍDO DE GESTÃO DE FROTA (v3.0 Architecture)")
    print("="*70)

    # Initialize Infrastructure
    broker = MockMQTTBroker()
    store = StateStore()
    agent = AutonomousFleetAgent(broker, store)
    simulator = FleetSimulator(broker)

    # Monitor de Métricas (Rodando em paralelo)
    async def metrics_monitor():
        while True:
            await asyncio.sleep(1)
            metrics = agent.get_metrics()
            if metrics['total_processed'] > 0:
                print(f"\n📊 Throughput: {metrics['throughput_msgs_per_sec']:.2f} msg/s | Processados: {metrics['total_processed']} | Erros: {metrics['errors']}")
                # Mostra estado atual da frota
                states = {v: d['state'] for v, d in store.store.items()}
                if any(s != VehicleState.OPERATIONAL.value for s in states.values()):
                    print(f" ⚠️ ESTADO DA FROTA: {states}")

    # Executa Simulação e Monitor concurrently
    await asyncio.gather(
        simulator.run(duration_sec=8), # Roda por 8 segundos
        metrics_monitor()
    )

    print("\n" + "="*70)
    print("🏁 SIMULAÇÃO ENCERRADA. Resumo Final do Estado Store:")
    for vid, data in store.store.items():
        if data['state'] != VehicleState.OPERATIONAL.value:
            print(f" 🚨 {vid}: {data['state']} (Histórico: {len(data['history'])} transições)")
        else:
            print(f" ✅ {vid}: {data['state']}")

    print("\n💾 Logs de auditoria e comandos foram roteados pelo broker (simulado).")
    print("Esta arquitetura suporta escalamento horizontal infinito dos agentes.")

if __name__ == "__main__":
    # Use await main() directly in environments like Colab where an event loop is already running
    # or use asyncio.run(main()) in a script where no event loop is active.
    import nest_asyncio
    nest_asyncio.apply()
    await main()

🌐 INICIANDO SISTEMA DISTRIBUÍDO DE GESTÃO DE FROTA (v3.0 Architecture)
📡 Subscriber registrado no tópico: telemetry/data
🤖 Iniciado e ouvindo stream de telemetria...
🚛 Iniciando frota com 10 veículos por 8s...

📊 Throughput: 45.95 msg/s | Processados: 46 | Erros: 0

📊 Throughput: 42.45 msg/s | Processados: 85 | Erros: 0
 ⚠️ ESTADO DA FROTA: {'V-001': 'OPERATIONAL', 'V-002': 'OPERATIONAL', 'V-003': 'OPERATIONAL', 'V-004': 'CRITICAL_STOP', 'V-005': 'OPERATIONAL', 'V-006': 'OPERATIONAL', 'V-007': 'OPERATIONAL', 'V-008': 'OPERATIONAL', 'V-009': 'OPERATIONAL', 'V-010': 'OPERATIONAL'}

📊 Throughput: 41.62 msg/s | Processados: 125 | Erros: 0
 ⚠️ ESTADO DA FROTA: {'V-001': 'OPERATIONAL', 'V-002': 'OPERATIONAL', 'V-003': 'OPERATIONAL', 'V-004': 'CRITICAL_STOP', 'V-005': 'OPERATIONAL', 'V-006': 'OPERATIONAL', 'V-007': 'OPERATIONAL', 'V-008': 'OPERATIONAL', 'V-009': 'OPERATIONAL', 'V-010': 'OPERATIONAL'}

📊 Throughput: 40.96 msg/s | Processados: 164 | Erros: 0
 ⚠️ ESTADO DA FROTA: {'V-001': 'OPER